# W5 作業｜用 pandas 清理與摘要失智症資料
**智慧醫療與機器學習基礎**　臺北醫學大學 智慧醫療學士學位學程　授課教師：王三源　課程助教：古珉瑄（醫資所，m610115009@tmu.edu.tw）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wang3yuan/FIHML/blob/main/homework/W05_HW.ipynb)

**開始前**：先「檔案 → 在雲端硬碟中儲存複本」，在自己的複本上作答，關掉分頁才不會不見。
**繳交**：完成後「檔案 → 下載 → 下載 .ipynb」，上傳至 Im@TMU。**期限：10/11（日）23:59**
**檔名**：`W05_HW_學號_姓名.ipynb`

### 資料
改編自 Kaggle [Alzheimer's Disease Dataset](https://www.kaggle.com/datasets/rabieelkharoua/alzheimers-disease-dataset)
（Rabie El Kharoua, 2024，CC BY 4.0）。原資料為合成資料；本課程把它拆成兩張表，並加入了髒資料。

| 檔案 | 內容 |
|---|---|
| `patients_raw.csv` | 病人基本資料、生活型態、病史（`PatientID` 為 key） |
| `clinical_raw.csv` | 臨床量測（血壓、膽固醇、MMSE、ADL……）與診斷 `Diagnosis` |

主要欄位說明：`Gender` 0 = 男、1 = 女；`MMSE` 簡易智能量表 0–30（越低越差）；`FunctionalAssessment`、`ADL` 0–10（越低越差）；`SleepQuality` 4–10；`Diagnosis` AD = 阿茲海默症。

### 執行環境
本教材已在 **Python 3.12＋pandas 2.2.3** 與 **pandas 3.0.6** 兩個版本測試通過。Colab 的預設版本會隨時間更新，執行下面第一格會印出你目前的版本。
- pandas 2.x：文字欄位的型別顯示為 `object`
- pandas 3.x：文字欄位的型別顯示為 `str`，意思相同，不影響本教材的程式

### 評分標準（100 分）
| 項目 | 配分 | 評分重點 |
|---|---|---|
| 1. 清理兩張表 | 40 | 每個問題都有處理、處理前後有印出確認 |
| 2. 合併 | 15 | 比較不同 `how` 的列數，並說明選擇理由 |
| 3. 比較表與交叉表 | 25 | 結果正確、表格可讀 |
| 4. 兩個發現 | 10 | 數字＋解讀＋限制 |
| 5. 資料字典 | 10 | 議題（Miro 上任一組或自行發想）、選題理由、≥ 5 欄且欄位完整、最難取得的資料 |

> 💡 程式要能**從頭到尾執行不出錯**（執行階段 → 重新啟動並全部執行），助教會這樣檢查。
> 💡 用 `map` 轉換編碼的格，重複執行可能會讓整欄變成缺值（因為已經轉過的值不在對照表裡）。結果怪怪的時候，請從頭全部執行一次。

In [ ]:
import pandas as pd
import numpy as np

HW = "https://raw.githubusercontent.com/wang3yuan/FIHML/main/data/hw/"
print("pandas 版本：", pd.__version__)
pat = pd.read_csv(HW + "patients_raw.csv")
cli = pd.read_csv(HW + "clinical_raw.csv")
print(pat.shape, cli.shape)

---
## 1. 清理（40 分）
### 1-1 先檢查（不計分，但請做）
用 `info()`、`describe()`、`value_counts()` 找出兩張表的問題，在下方文字格列出你發現的問題。

In [ ]:
pat.info()
cli.info()
# TODO：繼續用 describe()、value_counts() 檢查其他欄位


> 我發現的問題：
> -
> - 

### 1-2 清理 `pat`（20 分）
請完成：
1. 刪除完全重複的列，並確認 `PatientID` 唯一
2. `Gender` 統一為 0（男）／1（女）的整數
3. `Smoking` 統一為 0／1 的整數
4. `BMI` 去掉單位文字並轉為數值
5. `Age` 範圍檢查：不合理的值設為 `NaN`（本資料病人年齡應介於 60–90 歲）
6. 印出清理後每欄的缺值數

⚠️ **注意**：`Gender`、`Smoking` 混了文字（例如 `Male`、`Yes`），所以整欄會被讀成文字，裡面的 0、1 其實是**字串** `"0"`、`"1"`。
寫對照表前先用 `value_counts()` 看清楚；轉換後一定要檢查有沒有變成 `NaN`（`isna().sum()`）。

In [ ]:
# TODO


### 1-3 清理 `cli`（20 分）
請完成：
1. 刪除完全重複的列
2. `MMSE` 中的 −1 代表「未施測」，換成 `NaN`
3. `VisitDate` 有兩種日期格式（`2023-09-06` 與 `08/11/2023`；斜線格式是美式的「月/日/年」），轉成日期型別（提示：`format="mixed"`）
4. 檢查血壓：收縮壓（`SystolicBP`）應該大於舒張壓（`DiastolicBP`）。有幾筆不符合？請在下方文字格回答：臨床上可能嗎？你會怎麼處理？
5. `Diagnosis` 轉成 1（AD）／0（Non-AD）
6. 印出清理後每欄的缺值數

In [ ]:
# TODO


> 血壓的檢查結果與處理方式：
> 

---
## 2. 合併（15 分）
1. 分別用 `how="inner"`、`how="left"`（以 `pat` 為左表）、`how="outer"` 合併，印出各自的列數。
2. 有幾位病人在 `pat` 有、`cli` 沒有？有幾筆在 `cli` 有、`pat` 沒有？
   提示：`merge(..., how="outer", indicator=True)` 會多一欄 `_merge`，標示每一列來自 `left_only`、`right_only` 或 `both`，再對它做 `value_counts()`。
3. 你最後選擇哪一種 `how`？為什麼？（寫在文字格）

In [ ]:
# TODO


> 我選擇的 `how` 與理由：

---
## 3. 比較表與交叉表（25 分）
1. 依 `Diagnosis` 分組，計算 `Age`、`MMSE`、`FunctionalAssessment`、`ADL`、`SleepQuality`、`BMI` 的平均（四捨五入到小數 2 位）。
2. 用 `pd.cut` 把 `MMSE` 分成三級：≤ 17（中重度）、17 以上到 23（輕度）、23 以上到 30（正常）。MMSE 有小數，切點請用 `bins=[-0.1, 17, 23, 30]`，做 MMSE 分級 × `Diagnosis` 的交叉表（每列加總為 1）。
3. 做 `MemoryComplaints` × `Diagnosis` 的交叉表（每列加總為 1）。

In [ ]:
# TODO


In [ ]:
# TODO


In [ ]:
# TODO


---
## 4. 兩個發現（10 分）
根據第 3 題的結果，寫下**兩個**發現。每個發現都要包含：**數字**、**解讀**、**限制**（例如：這是描述統計，不代表因果；這份資料是合成資料……）。

> 發現 1：
>
> 發現 2：

---
## 5. 照護議題的資料字典（10 分）
從**課程 Miro 白板**（連結請見 Im@TMU）上各組的議題中**選一個**，也可以**自己繼續發想一個新的議題**，不限定是自己小組的題目。

**(a)** 議題名稱與來源（Miro 上哪一組的議題，或自己發想）：

**(b)** 為什麼選這個議題？（1–2 句）

**(c)** 資料字典：列出 **5 個以上**需要的資料欄位。

| 欄位名稱 | 意義 | 型別／單位 | 誰記錄？多久記一次？ | 是否為個資？ | 確診前可取得？ |
|---|---|---|---|---|---|
| | | | | | |
| | | | | | |
| | | | | | |
| | | | | | |
| | | | | | |

⚠️ 若要預測某個診斷，**確診之後才產生的資料**（例如失智症用藥、CDR 分期）不能作為模型輸入。

**(d)** 上表中，你認為**最難取得**的是哪一項資料？為什麼？